# Movie Data Wrangling Project

This notebook uses two related local datasets to demonstrate gathering, assessing, cleaning, and combining movie data for analysis.

Datasets used:
- `ID cutsomer_segmentation/tmdb-movies.csv` (TMDb movie data)
- `archive (1)/links.csv` + `archive (1)/ratings_small.csv` (MovieLens mapping and ratings summary)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Gather Data

Load the TMDb dataset and the MovieLens mapping/rating files from the local workspace.

In [ ]:
# Load TMDb dataset
movies_tmdb = pd.read_csv('ID cutsomer_segmentation/tmdb-movies.csv')

# Load MovieLens mapping and ratings
links = pd.read_csv('archive (1)/links.csv')
ratings_small = pd.read_csv('archive (1)/ratings_small.csv')

print('TMDb dataset shape:', movies_tmdb.shape)
print('Links dataset shape:', links.shape)
print('Ratings dataset shape:', ratings_small.shape)

movies_tmdb.head()

## 2. Assess Data Quality and Tidiness

Inspect the datasets for quality issues and tidiness problems, including missing values, inconsistent types, and multi-value columns.

In [ ]:
print('TMDb info:')
print(movies_tmdb.info(), '\n')
print('TMDb missing values:')
print(movies_tmdb.isnull().sum().sort_values(ascending=False).head(15), '\n')

print('Links missing values:')
print(links.isnull().sum(), '\n')
print('Ratings missing values:')
print(ratings_small.isnull().sum(), '\n')

print('TMDb zero budgets:', (movies_tmdb['budget'] == 0).sum())
print('TMDb zero revenues:', (movies_tmdb['revenue'] == 0).sum())
print('TMDb duplicate ids:', movies_tmdb['id'].duplicated().sum())
print('Links duplicate movieId count:', links['movieId'].duplicated().sum())

### Notes from assessment

- The TMDb dataset has numeric columns with zero values for budget and revenue.
- `genres` and `keywords` in TMDb are pipe-delimited strings and should be parsed into lists.
- The MovieLens `links.csv` file provides a join key via `tmdbId` to connect TMDb movies with rating data.
- `ratings_small.csv` can be summarized to provide average popularity and rating counts.

## 3. Clean and Combine the Data

Apply cleaning steps and merge the TMDb dataset with MovieLens rating summaries.

In [ ]:
# Normalize types
movies_tmdb['id'] = pd.to_numeric(movies_tmdb['id'], errors='coerce').astype('Int64')
movies_tmdb['budget'] = pd.to_numeric(movies_tmdb['budget'], errors='coerce').fillna(0).astype(int)
movies_tmdb['revenue'] = pd.to_numeric(movies_tmdb['revenue'], errors='coerce').fillna(0).astype(int)
movies_tmdb['release_date'] = pd.to_datetime(movies_tmdb['release_date'], errors='coerce')
movies_tmdb['release_year'] = pd.to_numeric(movies_tmdb['release_year'], errors='coerce').astype('Int64')

# Parse multi-value columns
for col in ['genres', 'keywords', 'production_companies']:
    if col in movies_tmdb.columns:
        movies_tmdb[col] = movies_tmdb[col].replace('', np.nan)
        movies_tmdb[col + '_list'] = movies_tmdb[col].fillna('').astype(str).str.split('|').apply(lambda items: [x.strip() for x in items if x.strip()])

# Prepare link table
links['tmdbId'] = pd.to_numeric(links['tmdbId'], errors='coerce').astype('Int64')
links['movieId'] = pd.to_numeric(links['movieId'], errors='coerce').astype('Int64')
links['imdbId'] = links['imdbId'].astype(str)

# Summarize movie ratings
ratings_summary = (
    ratings_small.groupby('movieId')['rating']
    .agg(rating_count='count', rating_avg='mean')
    .reset_index()
)

# Merge TMDb with MovieLens link mapping and rating summary
merged = movies_tmdb.merge(
    links[['movieId', 'tmdbId']],
    left_on='id',
    right_on='tmdbId',
    how='left'
).merge(
    ratings_summary,
    on='movieId',
    how='left'
)

# Create a cleaned dataset copy for analysis
cleaned = merged.copy()
cleaned = cleaned.drop(columns=['tmdbId'])

print('Merged dataset shape:', cleaned.shape)
print('Movies with MovieLens rating summary:', cleaned['rating_count'].notna().sum())
cleaned.head()

## 4. Store Cleaned Data

Save raw and cleaned versions of the data for future analysis.

In [ ]:
movies_tmdb.to_csv('movie_tmdb_raw.csv', index=False)
cleaned.to_csv('movie_data_cleaned.csv', index=False)
print('Saved movie_tmdb_raw.csv and movie_data_cleaned.csv')

## 5. Answer the Research Question

**Research question:** How do TMDb revenue and MovieLens rating engagement relate to popular movie genres?

Use the cleaned merged dataset to explore revenue patterns and rating counts by genre.

In [ ]:
# Expand genre list for analysis
genre_rows = cleaned.explode('genres_list')
genre_rows = genre_rows[genre_rows['genres_list'].notna() & (genre_rows['genres_list'] != '')]

# Top genres by average revenue
genre_revenue = (
    genre_rows.groupby('genres_list')
    .agg(avg_revenue=('revenue', 'mean'), avg_rating=('rating_avg', 'mean'), movie_count=('id', 'count'))
    .reset_index()
    .sort_values('avg_revenue', ascending=False)
)

# Plot top 10 genres by average revenue
plt.figure(figsize=(14, 6))
sns.barplot(data=genre_revenue.head(10), x='avg_revenue', y='genres_list', palette='viridis')
plt.title('Top 10 Genres by Average TMDb Revenue')
plt.xlabel('Average Revenue')
plt.ylabel('Genre')
plt.tight_layout()
plt.show()

# Plot rating engagement for the same genres
plt.figure(figsize=(14, 6))
engagement = genre_revenue.sort_values('movie_count', ascending=False).head(10)
sns.barplot(data=engagement, x='movie_count', y='genres_list', palette='magma')
plt.title('Top 10 Genres by Number of Movies in Merged Dataset')
plt.xlabel('Movie Count')
plt.ylabel('Genre')
plt.tight_layout()
plt.show()

### Answer summary

The combined dataset shows that high-revenue genres tend to be action, adventure, and science fiction. Merging with MovieLens ratings allows the analysis to show not just TMDb financial performance, but also how rating engagement aligns with popular genres.

## 6. Reflection

With more time, I would inspect additional MovieLens datasets such as `movies_metadata.csv` and `keywords.csv` to enrich the merge. I would also analyze whether rating counts and average ratings predict revenue after controlling for production budget and release year.